In [46]:
import psycopg2
import pandas as pd
import os

# 1. Setup Environment
output_dir = './data'
os.makedirs(output_dir, exist_ok=True)

conn_params = {
    "dbname": "musicbrainz_db",
    "user": "musicbrainz",
    "password": "musicbrainz",
    "host": "localhost",
    "port": "5432"
}

def export_clean_parquet(name, query):
    print(f"🚀 Exporting {name}...")
    try:
        # Fetch using standard psycopg2 to avoid the 'pandas.period' error
        conn = psycopg2.connect(**conn_params)
        cur = conn.cursor()
        cur.execute(query)
        data = cur.fetchall()
        colnames = [desc[0] for desc in cur.description]
        cur.close()
        conn.close()

        df = pd.DataFrame(data, columns=colnames)
        
        # TYPE SCRUB: Convert 'object' types to 'string' 
        # This prevents the pyarrow 'pandas.period' conflict we fixed earlier
        for col in df.columns:
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str)

        file_path = os.path.join(output_dir, f"{name}.parquet")
        df.to_parquet(file_path, index=False, engine='pyarrow')
        print(f"✅ Saved {len(df)} rows to {file_path}")
        
    except Exception as e:
        print(f"❌ Error during export of {name}: {e}")

# --- 1. THE ARTIST LOOKUP (Unique IDs to Names) ---
q_artists = """
SELECT DISTINCT
    a.gid AS artist_mbid, 
    a.name AS artist_name
FROM musicbrainz.artist a
JOIN musicbrainz.artist_credit_name acn ON a.id = acn.artist
JOIN musicbrainz.release_group rg ON acn.artist_credit = rg.artist_credit
JOIN musicbrainz.release_group_primary_type pt ON rg.type = pt.id
WHERE a.name != 'Various Artists'
  AND pt.name = 'Album'
LIMIT 1000000;
"""

# --- 2. THE ALBUM CORE (Album Name linked to Artist ID) ---
q_albums = """
SELECT 
    rg.gid AS album_mbid, 
    rg.name AS album_name, 
    a.gid AS artist_mbid
FROM musicbrainz.release_group rg
JOIN musicbrainz.artist_credit_name acn ON rg.artist_credit = acn.artist_credit
JOIN musicbrainz.artist a ON acn.artist = a.id
JOIN musicbrainz.release_group_primary_type pt ON rg.type = pt.id 
WHERE a.name != 'Various Artists'
  AND pt.name = 'Album'
  AND rg.gid IN (
      SELECT rg_inner.gid
      FROM musicbrainz.release_group_tag rgt_inner
      JOIN musicbrainz.release_group rg_inner ON rgt_inner.release_group = rg_inner.id
      GROUP BY rg_inner.gid
      HAVING COUNT(rgt_inner.tag) >= 2
  )
LIMIT 1000000;
"""

# --- 3. THE ALBUM TAGS (Features for the Album ID) ---
q_tags = """
SELECT 
    rg.gid AS album_mbid, 
    t.name AS tag, 
    rgt.count AS weight
FROM musicbrainz.release_group_tag rgt
JOIN musicbrainz.tag t ON rgt.tag = t.id
JOIN musicbrainz.release_group rg ON rgt.release_group = rg.id
WHERE rg.gid IN (
    -- Subquery: Find albums that have at least 2 distinct tags
    SELECT rg_inner.gid
    FROM musicbrainz.release_group_tag rgt_inner
    JOIN musicbrainz.release_group rg_inner ON rgt_inner.release_group = rg_inner.id
    GROUP BY rg_inner.gid
    HAVING COUNT(rgt_inner.tag) >= 2
)
AND rgt.count > 1 -- Keeping the weight filter as well
ORDER BY weight DESC;
"""

# Run the Extraction
if __name__ == "__main__":
    export_clean_parquet("artists", q_artists)
    export_clean_parquet("albums", q_albums)
    export_clean_parquet("album_tags", q_tags)
    
    print("\n🎉 Feature Store Build Complete!")
    print("Files ready in ./data/: artists.parquet, albums.parquet, album_tags.parquet")

🚀 Exporting artists...
✅ Saved 689596 rows to ./data/artists.parquet
🚀 Exporting albums...
✅ Saved 759855 rows to ./data/albums.parquet
🚀 Exporting album_tags...
✅ Saved 341188 rows to ./data/album_tags.parquet

🎉 Feature Store Build Complete!
Files ready in ./data/: artists.parquet, albums.parquet, album_tags.parquet


In [47]:
import pandas as pd

# Load everything
albums = pd.read_parquet('./data/albums.parquet')
tags = pd.read_parquet('./data/album_tags.parquet')
artists = pd.read_parquet('./data/artists.parquet')

# Merge on the fly
# Get Album + Tags
df = albums.merge(tags, on='album_mbid', how='inner')
# Get Artist Name
df = df.merge(artists, on='artist_mbid', how='left')

print(df.head())

                             album_mbid      album_name  \
0  64b5b86e-1109-3a00-b5c0-4eeda8a7faa2  Loop Bites Dog   
1  64b5b86e-1109-3a00-b5c0-4eeda8a7faa2  Loop Bites Dog   
2  c09bf526-416d-3f9c-8291-8794cef83d87   Red Dirt Girl   
3  6cec43d7-9bcd-3530-9ee0-a2c4091da339        Ambrosia   
4  6cec43d7-9bcd-3530-9ee0-a2c4091da339        Ambrosia   

                            artist_mbid         tag  weight  \
0  375d23d9-d767-4a8e-bd0c-9fae1e26e51e  electronic       3   
1  375d23d9-d767-4a8e-bd0c-9fae1e26e51e   downtempo       2   
2  35ef61ca-43db-4772-ba27-0489e9ebcb69        folk       3   
3  3376bc9f-c766-4a7c-8792-c1c938587e00   downtempo       2   
4  3376bc9f-c766-4a7c-8792-c1c938587e00  electronic       2   

           artist_name  
0            Loop Guru  
1            Loop Guru  
2       Emmylou Harris  
3  A Reminiscent Drive  
4  A Reminiscent Drive  


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 225711 entries, 0 to 225710
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   album_mbid   225711 non-null  str  
 1   album_name   225711 non-null  str  
 2   artist_mbid  225711 non-null  str  
 3   tag          225711 non-null  str  
 4   weight       225711 non-null  int64
 5   artist_name  225711 non-null  str  
dtypes: int64(1), str(5)
memory usage: 34.7 MB


In [54]:
artists.info()

<class 'pandas.DataFrame'>
RangeIndex: 689596 entries, 0 to 689595
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   artist_mbid  689596 non-null  str  
 1   artist_name  689596 non-null  str  
dtypes: str(2)
memory usage: 42.7 MB


In [53]:
albums.info()

<class 'pandas.DataFrame'>
RangeIndex: 759855 entries, 0 to 759854
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   album_mbid   759855 non-null  str  
 1   album_name   759855 non-null  str  
 2   artist_mbid  759855 non-null  str  
dtypes: str(3)
memory usage: 84.0 MB


In [52]:
album_tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 356150 entries, 0 to 356149
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   album_mbid  356150 non-null  str  
 1   tag         356150 non-null  str  
 2   weight      356150 non-null  int64
dtypes: int64(1), str(2)
memory usage: 23.3 MB


In [51]:
tags_df = pd.read_parquet('./data/album_tags.parquet')
print(tags_df['tag'].value_counts().head(20))

tag
electronic          54658
rock                27136
pop                 10955
techno               8747
ambient              8352
experimental         7681
house                7535
hip hop              6376
jazz                 5727
punk                 5725
alternative rock     5702
pop rock             5531
electro              5447
indie rock           4978
downtempo            4584
synth-pop            4543
classical            4367
black metal          4158
industrial           3717
trance               3368
Name: count, dtype: int64
